In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter, NullLocator, MaxNLocator
from pathlib import Path
from matplotlib import font_manager as fm
from matplotlib.colors import Normalize
from rasterio.warp import reproject
import sys, pathlib, importlib

sys.path.append("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
import Robyn_paper_2_defs
import Robyn_river_floods


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_dir = base_path / "dphil_paper_2/results"

In [ ]:
J2USD = 1.0 / 150.0  # convert J$ → US$


# Read in damage reduction

In [ ]:
damage_reduction_min = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_min.tif"
damage_reduction_max = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_max.tif"
damage_reduction_max_smoothed = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_max_smoothed.tif"
damage_reduction_min_smoothed = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_min_smoothed.tif"

In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

# LOG MAP J$

In [ ]:
with rasterio.open(damage_reduction_min_smoothed) as src:
    arr    = src.read(1, masked=True) 
    
arr.sum() * 1e-9

In [ ]:
# # --- Fig 3(c) — log scale, using rcParams for font sizes --------------------

# with rasterio.open(damage_reduction_max) as src:
#     arr    = src.read(1, masked=True)      # respects NoData
#     extent = plotting_extent(src)

# # data JD → thousands of USD; mask nonpositives
# data_kusd = np.array(arr, dtype="float64") * J2USD * 1e-3
# data_kusd[arr.mask | (data_kusd <= 0)] = np.nan

# # TODO replace data_bil below here with data_kusd ; delete any later unit conversions

# # robust bounds (2–98%) for LogNorm
# tiny = 1e-9
# pos = np.asarray(data_kusd[(~np.isnan(data_kusd)) & (data_kusd > 0)])
# if pos.size == 0:
#     raise ValueError("No positive values to plot (all zeros/NaNs).")

# lo, hi = np.percentile(pos, [2, 98])
# lo = max(float(lo), tiny)
# hi = float(hi if hi > lo else lo * 1.01)

# norm = LogNorm(vmin=lo, vmax=hi)  # single source of truth

# # colormap
# cmap = mpl.colormaps["Greens"].copy()
# cmap.set_bad((0, 0, 0, 0))  # transparent NoData

# # choose human-friendly units given vmax (still in billions)
# def pick_unit_for_billions(hi_bil: float):
#     if hi_bil >= 0.5:      return 1.0,  "J$ billions"   # keep billions
#     if hi_bil >= 0.005:    return 1e3, "J$ millions"    # B → M
#     return 1e6, "J$ thousands"                          # B → k

# scale, unit_label = pick_unit_for_billions(hi)

# with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
#     # pull sizes from rcParams so you don't need hardcoded TITLE_FS/LABEL_FS/TICK_FS
#     TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
#     LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
#     TICK_FS  = mpl.rcParams.get("xtick.labelsize", 5.5)

#     fig, ax = plt.subplots(figsize=(Robyn_river_floods.mm_to_in(90), Robyn_river_floods.mm_to_in(60)))
#     ax.set_axis_off()

#     # raster
#     im = ax.imshow(data_kusd, norm=norm, cmap=cmap, extent=extent, origin="upper", interpolation="nearest")

#     # Jamaica outline (white casing + black)
#     try:
#         outline_geom = jamaica_boundary.union_all()
#     except AttributeError:
#         outline_geom = jamaica_boundary.unary_union
#     outline_gdf = gpd.GeoSeries([outline_geom], crs=jamaica_boundary.crs)
#     outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
#     outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)

#     # --- Colorbar: decade ticks, readable labels in chosen units ---
#     sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
#     # powers of 10 within [lo, hi]
#     lo_pow = int(np.floor(np.log10(lo)))
#     hi_pow = int(np.ceil(np.log10(hi)))
#     ticks = (10.0 ** np.arange(lo_pow, hi_pow + 1))
#     ticks = ticks[(ticks >= lo) & (ticks <= hi)]
#     cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.015, ticks=ticks)

#     def tick_fmt(x, _):
#         v = x * scale  # convert from billions to chosen unit
#         if scale == 1.0:         # billions
#             return f"{v:.2f}" if v < 10 else f"{v:.1f}"
#         if scale == 1e3:         # millions
#             return f"{v:.1f}" if v < 10 else f"{v:.0f}"
#         else:                    # thousands
#             return f"{v:.2f}" if v < 1 else (f"{v:.1f}" if v < 10 else f"{v:.0f}")
#     cbar.ax.yaxis.set_major_formatter(FuncFormatter(tick_fmt))
#     cbar.minorticks_off()
#     cbar.ax.yaxis.set_minor_locator(NullLocator())
#     cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)
#     cbar.set_label(f"Avoided damages ({unit_label})", fontsize=LABEL_FS)

#     # --- Scale bar + North arrow --------------------------------------------------
#     Robyn_paper_2_defs.draw_scale_bar(ax, location=(0.88, 0.78), length_km=20, linewidth=0.6,
#                   label_offset=0.02, km_offset=0.01)
#     Robyn_paper_2_defs.draw_north_arrow(ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

    
#     # higher title (use suptitle)
#     fig.subplots_adjust(top=0.96)  # leave a bit more headroom
#     fig.suptitle(
#         "Fig 3(c) Candidates for forest restoration with highest avoided\nexpected annual damages (log scale)",
#         fontsize=TITLE_FS, y=0.985
#     )

#     plt.show()
#     out_png = output_dir / "forest_avoided_damages_log_thousands.png"
#     fig.savefig(out_png, dpi=600, bbox_inches="tight")  # RGB, ≥300 dpi
#     print("Saved:", out_png)

# NON-LOG MAP USD

In [ ]:
with rasterio.open(damage_reduction_min) as src:
    arr    = src.read(1, masked=True) 
    
arr.sum() * 1e-9

In [ ]:
# # === Fig 3(b) — Candidates based on avoided EADs (plain USD colorbar) ========
# damage_reduction_max = globals().get("damage_reduction_max")  # path to raster
# if damage_reduction_max is None:
#     raise ValueError("Set `damage_reduction_max` to the path of your raster.")

# # --- read raster --------------------------------------------------------------
# with rasterio.open(damage_reduction_max) as src:
#     arr    = src.read(1, masked=True)   # masked array; respects NoData
#     b      = src.bounds
#     extent = (b.left, b.right, b.bottom, b.top)
#     r_crs  = src.crs

# # --- Convert J$ → USD; mask non-positives ------------------------------------
# data_usd = np.array(arr, dtype="float64") * J2USD
# data_usd[arr.mask | (data_usd <= 0)] = np.nan

# # --- Robust upper bound for linear scale (2–98th pct), with fallbacks --------
# pos = data_usd[~np.isnan(data_usd)]
# if pos.size == 0:
#     raise ValueError("No positive values to plot (all zeros/NaNs).")

# lo = 0.0
# hi = np.percentile(pos, 98)
# if not np.isfinite(hi) or hi <= 0:
#     hi = float(np.nanmax(pos)) if np.isfinite(np.nanmax(pos)) else 1.0

# # --- Normalization (LINEAR) in USD -------------------------------------------
# norm = mpl.colors.Normalize(vmin=lo, vmax=hi)

# # --- Colormap (keep NoData transparent) --------------------------------------
# cmap = mpl.colormaps["Greens"].copy()
# cmap.set_bad((0, 0, 0, 0))  # transparent

# with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
#     TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
#     LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
#     TICK_FS  = mpl.rcParams.get("ytick.labelsize", mpl.rcParams.get("xtick.labelsize", 5.5))

#     fig, ax = plt.subplots(figsize=(Robyn_river_floods.mm_to_in(90), Robyn_river_floods.mm_to_in(60)))
#     ax.set_axis_off()

#     # --- Raster ----------------------------------------------------------------
#     im = ax.imshow(
#         data_usd, norm=norm, cmap=cmap, extent=extent,
#         origin="upper", interpolation="nearest"
#     )

#     # --- Jamaica outline (white casing + black), reprojected to raster CRS ----
#     try:
#         jamaica_boundary = globals()["jamaica_boundary"]
#         outline_geom = jamaica_boundary.union_all() if hasattr(jamaica_boundary, "union_all") else jamaica_boundary.unary_union
#         outline_gdf = gpd.GeoSeries([outline_geom], crs=getattr(jamaica_boundary, "crs", None))
#         if getattr(outline_gdf, "crs", None) is not None and r_crs is not None:
#             outline_gdf = outline_gdf.to_crs(r_crs)
#         outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
#         outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)
#     except Exception as e:
#         print("Outline skipped:", e)
#         outline_gdf = None

#     # --- Colorbar: ticks in **plain USD** -------------------------------------
#     sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
#     cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.015)
#     cbar.ax.yaxis.set_minor_locator(NullLocator())
#     cbar.outline.set_linewidth(0.35)

#     # Choose 4–6 nice ticks from 0 → hi in USD
#     locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
#     ticks_usd = locator.tick_values(0, hi)
#     ticks_usd = ticks_usd[(ticks_usd >= 0) & (ticks_usd <= hi + 1e-12)]
#     cbar.set_ticks(ticks_usd)

#     # Format: thousands separators (no unit suffix; title carries "US$")
#     def _fmt_usd(y_usd):
#         if y_usd < 10:
#             s = f"{y_usd:.2f}"
#         elif y_usd < 100:
#             s = f"{y_usd:.1f}"
#         else:
#             s = f"{y_usd:,.0f}"
#         return s

#     cbar.set_ticklabels([_fmt_usd(t) for t in ticks_usd])
#     cbar.set_label("Avoided damages (US$)", fontsize=LABEL_FS)
#     cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)

#     # --- Scale bar + North arrow --------------------------------------------------
#     Robyn_paper_2_defs.draw_scale_bar(ax, location=(0.88, 0.78), length_km=20, linewidth=0.6,
#                   label_offset=0.02, km_offset=0.01)
#     Robyn_paper_2_defs.draw_north_arrow(ax, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)


#     # --- Save ------------------------------------------------------------------
#     out_png = output_dir / "forest_avoided_damages_linear_USD_plain_units_check.png"
#     fig.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
#     plt.show()
#     print("Saved:", out_png)

In [ ]:
# === Fig. 3(b) — Candidates based on avoided EADs (plain USD), same size =====

damage_reduction_min = globals().get("damage_reduction_min")
if damage_reduction_min is None:
    raise ValueError("Set `damage_reduction_min` to the path of your raster.")


# --- read raster --------------------------------------------------------------
with rasterio.open(damage_reduction_min) as src:
    arr    = src.read(1, masked=True)
    b      = src.bounds
    extent = (b.left, b.right, b.bottom, b.top)
    r_crs  = src.crs

# --- Convert J$ → USD; mask non-positives ------------------------------------
data_usd = np.array(arr, dtype="float64") * J2USD
data_usd[arr.mask | (data_usd <= 0)] = np.nan

# --- Robust upper bound (match top map’s approach: 99.5th percentile) --------
pos = data_usd[~np.isnan(data_usd)]
if pos.size == 0:
    raise ValueError("No positive values to plot (all zeros/NaNs).")
hi = float(np.percentile(pos, 99.5))
if not np.isfinite(hi) or hi <= 0:
    hi = float(np.nanmax(pos)) if np.isfinite(np.nanmax(pos)) else 1.0

# --- Normalization (LINEAR) in USD -------------------------------------------
norm = mpl.colors.Normalize(vmin=0.0, vmax=hi)

# --- Colormap (NoData transparent) -------------------------------------------
cmap = mpl.colormaps["Greens"].copy()
cmap.set_bad((0, 0, 0, 0))

with mpl.rc_context(Robyn_paper_2_defs.NATURE_RC):
    TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
    LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
    TICK_FS  = mpl.rcParams.get("ytick.labelsize", mpl.rcParams.get("xtick.labelsize", 5.5))

    # Match the top map’s figure size exactly
    fig, ax = plt.subplots(figsize=(Robyn_river_floods.mm_to_in(90), Robyn_river_floods.mm_to_in(60)))
    ax.set_axis_off()

    # Raster
    im = ax.imshow(
        data_usd, norm=norm, cmap=cmap, extent=extent,
        origin="upper", interpolation="nearest"
    )

    # Jamaica outline (white casing + black), reprojected to raster CRS
    outline_gdf = None
    try:
        jamaica_boundary = globals()["jamaica_boundary"]
        outline = jamaica_boundary.union_all() if hasattr(jamaica_boundary, "union_all") else jamaica_boundary.unary_union
        outline_gdf = gpd.GeoSeries([outline], crs=getattr(jamaica_boundary, "crs", None))
        if getattr(outline_gdf, "crs", None) is not None and r_crs is not None:
            outline_gdf = outline_gdf.to_crs(r_crs)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)
    except Exception as e:
        print("Outline skipped:", e)
        outline_gdf = None

    # Colorbar (match top map: fraction/pad; plain USD ticks)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.024, pad=0.012)  # ← same as top map
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)

    locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
    ticks_usd = locator.tick_values(0, hi)
    ticks_usd = ticks_usd[(ticks_usd >= 0) & (ticks_usd <= hi + 1e-12)]
    cbar.set_ticks(ticks_usd)

    def _fmt_usd(y):
        if y < 10:
            return f"{y:.2f}"
        elif y < 100:
            return f"{y:.1f}"
        else:
            return f"{y:,.0f}"
    cbar.set_ticklabels([_fmt_usd(t) for t in ticks_usd])
    cbar.set_label("Avoided damages (US$)", fontsize=LABEL_FS)
    cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)

    # NOTE: removed scale bar & north arrow for this green map

    # Save (same export settings)
    out_png = output_dir / "figures/Fig_3b_forest_avoided_damages_linear_USD_plain_units_min_supplementary.png"
    fig.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Saved:", out_png)